In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler

df = pd.read_csv("../data/processed/telco_churn_clean.csv")
df.shape

(7043, 20)

In [2]:
X = df.drop(columns="Churn")
y = df["Churn"]
print("X:", X.shape, " y:", y.shape)

X: (7043, 19)  y: (7043,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,
stratify=y,random_state=42)

print("train:",X_train.shape[0], " test:", X_test.shape[0] )
print("churn rate train :%.4f  test:%.4f " %(y_train.mean(), y_test.mean()))


train: 5634  test: 1409
churn rate train :0.2654  test:0.2654 


In [4]:
numeric_features = make_column_selector(dtype_include="number")
categorical_features = make_column_selector(dtype_exclude="number")

print("numeric:", len(numeric_features(X)), "categorical:",len(categorical_features(X)))

numeric: 4 categorical: 15


In [5]:
preprocessor = ColumnTransformer([("num", StandardScaler() , numeric_features),
("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

In [6]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier" , LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

print("features after encoding:", model.named_steps["preprocessor"].transform(X_train).shape[1])

features after encoding: 45


In [12]:
accuracy = model.score(X_test, y_test)
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

print("Logistic regression: %.4f" % accuracy)
print("Always predict 'No': %.4f" % dummy.score(X_test, y_test))

print("improvement: %.1f percentage points" % ((accuracy - dummy.score(X_test,y_test))*100))

Logistic regression: 0.8055
Always predict 'No': 0.7346
improvement: 7.1 percentage points
